In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import loompy as lp

In [3]:
adata1=sc.read_h5ad("/home/julia/data/placodes/ANDYs_notebboks/h5ad/E9_E10_E11_QC_SCRUBLET_020.h5ad")

In [4]:
adata_sub=sc.read_h5ad("/home/julia/data/placodes/ANDYs_notebboks/Velocity_ANDY_23_09/Sub_8_13_4_UMAP_last_1.h5ad")

In [5]:
# Select cells from adata that are present in adata_sub
common_cells = adata1.obs_names.intersection(adata_sub.obs_names)

adata = adata1[common_cells, :].copy()

print("Original:", adata1.shape)
print("Subset:", adata_sub.shape)
print("SCENIC:", adata.shape)

Original: (94890, 32285)
Subset: (2304, 32285)
SCENIC: (2304, 32285)


In [6]:
#adata=sc.read_h5ad('/home/julia/data/placodes/h5ad/11_normalized.h5ad')

In [6]:
adata.obs_names_make_unique()

In [7]:
#adata=sc.read_h5ad('/home/julia/data/placodes/noutbooks/h5ad_files/14_06_ALL_7_datasets_qc.h5ad')

In [8]:
adata

AnnData object with n_obs × n_vars = 2304 × 32285
    obs: 'day', 'plate', 'sample_id', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'doublet_020'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    layers: 'counts'

In [17]:
# path to unfiltered loom file (this will be created in the optional steps below)
#f_loom_path_unfilt = "placods_unfiltered.loom" # test dataset, n=500 cells

# # path to loom file with basic filtering applied (this will be created in the "initial filtering" step below). Optional.
f_loom_path_scenic = "All_filtered_scenic.loom"

# path to anndata object, which will be updated to store Scanpy results as they are generated below
f_anndata_path = "anndata.h5ad"

# path to pyscenic output
f_pyscenic_output = "All_pyscenic_output.loom"

# loom output, generated from a combination of Scanpy and pySCENIC results:
f_final_loom = 'All_scenic_integrated-output.loom'

In [18]:
# create basic row and column attributes for the loom file:
row_attrs = {
    "Gene": np.array(adata.var_names) ,
}
col_attrs = {
    "CellID": np.array(adata.obs_names) ,
    "nGene": np.array( np.sum(adata.X.transpose()>0 , axis=0)).flatten() ,
    "nUMI": np.array( np.sum(adata.X.transpose() , axis=0)).flatten() ,
}
lp.create( f_loom_path_scenic, adata.X.transpose(), row_attrs, col_attrs)

In [19]:
pwd

'/mnt/data/julia/placodes/ANDYs_notebboks/Scenic_neuro'

In [21]:
!/home/julia/miniconda3/envs/scenicplus/bin/pyscenic grn /mnt/data/julia/placodes/ANDYs_notebboks/Scenic_neuro/All_filtered_scenic.loom /home/julia/data/Scenic_database/Mouse/TF/allTFs_mm.txt -o adj_1.csv  --num_workers 30


2026-10-19 10:18:18,371 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2026-10-19 10:18:19,451 - pyscenic.cli.pyscenic - INFO - Inferring regulatory networks.
preparing dask client
parsing input
creating dask graph
30 partitions
computing dask graph
not shutting down client, client was created externally
finished
2026-10-19 11:48:17,854 - distributed.nanny - WARNING - Worker process still alive after 3.1999990844726565 seconds, killing

2026-10-19 11:48:17,994 - pyscenic.cli.pyscenic - INFO - Writing results to file.


In [22]:
adjacencies = pd.read_csv("adj_1.csv", index_col=False)

In [23]:
adjacencies

,TF,target,importance
0,Hmgb2,Arl6ip1,1.169357e+02
1,Hmgb2,Cks2,1.153843e+02
2,Hmgb2,Ccnb1,1.148512e+02
3,Peg3,Igfbp5,1.123483e+02
4,Hmgb2,Cdc20,1.118936e+02
...,...,...,...
6360817,Mtf1,Cln8,1.918575e-17
6360818,Ctbp1,Tmem127,1.873891e-17
6360819,Ubtf,Tacr1,1.680530e-17
6360820,Zfp697,Ikbip,1.611389e-17


In [24]:
f_tfs='/home/julia/data/Scenic_database/Mouse/TF/allTFs_mm.txt'

In [25]:
import glob
# ranking databases
f_db_glob = "/home/julia/data/Scenic_database/Mouse/Gene_based/*feather"
f_db_names = ' '.join( glob.glob(f_db_glob) )

In [26]:
!/home/julia/miniconda3/envs/scenicplus/bin/pyscenic ctx adj_1.csv \
    {f_db_names} \
    --annotations_fname /home/julia/data/Scenic_database/Mouse/Motif2TF/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl \
    --expression_mtx_fname /mnt/data/julia/placodes/ANDYs_notebboks/Scenic_neuro/All_filtered_scenic.loom \
    --output reg.csv \
    --mask_dropouts \
    --num_workers 30


2026-10-19 13:12:25,548 - pyscenic.cli.pyscenic - INFO - Creating modules.

2026-10-19 13:12:28,322 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2026-10-19 13:12:29,670 - pyscenic.utils - INFO - Calculating Pearson correlations.

2026-10-19 13:12:31,116 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [True].

2026-10-19 13:12:55,261 - pyscenic.utils - INFO - Creating modules.

2026-10-19 13:15:14,861 - pyscenic.cli.pyscenic - IN

In [27]:
!/home/julia/miniconda3/envs/scenicplus/bin/pyscenic aucell \
    {f_loom_path_scenic} \
    reg.csv \
    --output {f_pyscenic_output} \
    --num_workers 10


2026-10-19 13:30:31,132 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2026-10-19 13:30:32,199 - pyscenic.cli.pyscenic - INFO - Loading gene signatures.
Create regulons from a dataframe of enriched features.
Additional columns saved: []

2026-10-19 13:30:35,749 - pyscenic.cli.pyscenic - INFO - Calculating cellular enrichment.

2026-10-19 13:30:51,038 - pyscenic.cli.pyscenic - INFO - Writing results to file.
/home/julia/miniconda3/envs/scenicplus/lib/python3.8/site-packages/pyscenic/cli/utils.py:347: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for name, threshold in auc_thresholds.iteritems()


##